#### Problem 1
###### Write a computer program “package” that will take an initial spacecraft position and velocity vector, ro, vo, at an initial time to, and predict the future/past position and velocity of the spacecraft at an arbitrary time t

###### To accomplish this: 
###### a: Compute the orbital elements form an arbitrary position vector and velocity vector
###### b: Solve kepler's equation for an arbitrary time t to find the corresponding value of true anomaly
###### c: Using the orbital elements, specify the position and velocity vectors at the new value of true anomaly

In [ ]:
import numpy as np
from orbit_package.orbit_package import Orbit
import matplotlib.pyplot as plt

# Instantiation of orbit object
orb = Orbit()

# Part 1: 
mu = 4e5 # km^3/s^2
r0_vec = np.array([6e3,6e3,6e3]) # km
v0_vec = np.array([-5,5,0]) #km/s
elements = orb.orbital_elements(r0_vec,v0_vec,mu)
print(elements)

# Finding times to integrate at
period = elements['T']
integration_time = np.linspace(0,2*int(period),int(2*int(period)/60))

# Propagating the orbit
eph = orb.create_propagated_ephemeris(r0_vec,v0_vec,mu,integration_time)

# Plotting the resulting trajectory
from orbit_package.graphing_utils import Graph
gr = Graph()

print(eph.epoch)
gr.plot_eph_pos_vector(eph,earth=True)
gr.plot_eph_v_norm(eph)
gr.plot_eph_rv(eph)





In [ ]:
## Part two: Create orbit from incomplete orbital elements
from orbit_package.orbit_package import Orbit
from orbit_package.graphing_utils import Graph
import numpy as np
orb = Orbit()
gr = Graph()

mu = 4e5 # km^3/s^2
rp = 10000 #km
i = 135*np.pi/180 # rad
raan = 45*np.pi/180 # rad
argp = -90*np.pi/180 # rad

es = [0,0.25,0.5,0.75,0.99]

ephemerides = []
for e in es:
    # Creating elements dict
    elements = orb.fill_elements(mu,i,raan,argp,e,rp,0)
    T = elements['T']
    # Finding integratoin times
    integration_time = np.linspace(0,int(1.5*T),int(1.5*int(T)/60))

    # Finding an initial velocity and position vector to propagate
    r0_vec,v0_vec = orb.kep_to_cart(elements,mu)

    # Propagating r0 and v0 and creating ephemeris
    eph = orb.create_propagated_ephemeris(r0_vec,v0_vec,mu,integration_time)
    ephemerides.append(eph)

legend_values = ['e=0','e=0.25','e=0.5','e=0.75','e=0.99']
gr.plot_eph_pos_vector_listeph(ephemerides,legend_values=legend_values,earth=True)    

In [2]:
gr.plot_eph_pos_vector_listeph(ephemerides[0:4],legend_values=legend_values[0:4],earth=True)

In [1]:
## Part 2: Numerical Integration
# Load the autoreload extension
%load_ext autoreload
%autoreload 2
import numpy as np
from orbit_package.numerical import Integrator, ForcingFunction
from orbit_package.graphing_utils import Graph
from orbit_package.ephemeris import Ephemeris

# Instantiating all obejects
integrate = Integrator()
ff = ForcingFunction()
gr = Graph()

# Initial conditions and parameters
x0 = np.array([6e3,6e3,6e3,-5,5,0])
other = {"mu":4e5}
period = 17933.95497
tspan = [0,2*period]
integration_time = np.linspace(0,2*int(period),int(2*int(period)/60))

# Integrating
t,x = integrate.ode45(ff.twobody_nop_nodyn,x0,tspan,other,tval=integration_time)

eph = integrate.eph_from_propagation_results(t,x,"2BODY_NODYN")

gr.plot_eph_r_norm(eph)
gr.plot_eph_pos_vector(eph,earth=True)





In [1]:
# Comparing the orbits
import numpy as np
from orbit_package.orbit_package import Orbit
from orbit_package.numerical import Integrator, ForcingFunction



# Instantiation of orbit object
orb = Orbit()

## FINDING EPHEMERIS USING KEPLER'S EQUATION
mu = 4e5 # km^3/s^2
r0_vec = np.array([6e3,6e3,6e3]) # km
v0_vec = np.array([-5,5,0]) #km/s
elements = orb.orbital_elements(r0_vec,v0_vec,mu)

# Finding times to integrate at
period = elements['T']
integration_time = np.linspace(0,2*int(period),int(2*int(period)/60))

# Propagating the orbit
eph_kep = orb.create_propagated_ephemeris(r0_vec,v0_vec,mu,integration_time)

# Plotting the resulting trajectory
from orbit_package.graphing_utils import Graph
gr = Graph()

## INTEGRATING NUMERICALLY
integrate = Integrator()
ff = ForcingFunction()

# Initial conditions and parameters
x0 = np.array([6e3,6e3,6e3,-5,5,0])
other = {"mu":4e5}
tspan = [0,2*period]
integration_time = np.linspace(0,2*int(period),int(2*int(period)/60))

# Integrating
t,x = integrate.ode45(ff.twobody_nop_nodyn,x0,tspan,other,tval=integration_time)

eph_num = integrate.eph_from_propagation_results(t,x,"2BODY_NODYN")

ephemerides = [eph_kep,eph_num]
legend_values = ["Kepler's Eq","Numerical Integration"]

diff = orb.difference_eph(ephemerides)
gr.plot_eph_diff(ephemerides)

eph_osc = orb.fill_eph_osculating(eph_num,mu)

eph_full = orb.fill_eph_osculating(eph_kep,mu)
gr.plot_eph(eph_full,False,'Deg')

